# EKF Sensor Fusion Offline Analysis

This notebook reproduces the onboard Extended Kalman Filter (error-state, quaternion attitude)
by calling the **actual C functions** from `kalman_core.c` via `ctypes`.

**Workflow:**
1. Clone repo from GitHub into Colab
2. Compile `kalman_core.c` into a shared library with `make`
3. Load via ctypes and mirror the C structs in Python
4. Read time-series CSV data (IMU @ 200Hz, UWB @ 50Hz per anchor set)
5. Step through each measurement, calling predict/update exactly as the firmware does
6. Log and plot position, velocity, attitude, covariance over time

> You can edit C files directly in Colab and rerun the build cell to tune baked-in constants.


## 1. Setup: Clone Repository & Build Shared Library

Run the next cell to clone the firmware repository into Colab.

- If the repo is public, default settings work.
- If private, set `GITHUB_TOKEN` and use HTTPS token auth.
- Set `REPO_BRANCH` if you need a non-main branch.
- Place CSV files in `analysis/data/` inside the cloned repo (or upload there in Colab).


In [ ]:
import os
import shutil
import platform

# --- Git clone configuration ---
REPO_URL = 'https://github.com/aidanquandt/hybrid-localization-system.git'
REPO_BRANCH = 'eskf-analysis'
CLONE_PARENT = '/content'
CLONE_DIRNAME = 'hybrid-localization-system'
GITHUB_TOKEN = ''  # optional: set if repo is private

clone_target = os.path.join(CLONE_PARENT, CLONE_DIRNAME)
if os.path.exists(clone_target):
    shutil.rmtree(clone_target)

if GITHUB_TOKEN.strip():
    auth_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
else:
    auth_url = REPO_URL

cmd = f"git clone --depth 1 --branch {REPO_BRANCH} {auth_url} {clone_target}"
ret = os.system(cmd)
assert ret == 0, 'git clone failed. Check REPO_URL/REPO_BRANCH/token.'

REPO_ROOT = clone_target

# Locate analysis/ robustly (repo may be nested under one top-level folder)
def find_analysis_dir(root):
    direct = os.path.join(root, 'analysis')
    if os.path.isdir(direct):
        return direct

    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        # prune common heavy dirs for speed
        dirnames[:] = [d for d in dirnames if d not in {'.git', '.venv', '__pycache__', 'build', 'dist', 'node_modules'}]
        if os.path.basename(dirpath) == 'analysis':
            candidates.append(dirpath)

    if not candidates:
        return None

    # Prefer shallowest path under clone root
    candidates.sort(key=lambda p: p.count(os.sep))
    return candidates[0]

ANALYSIS_DIR = find_analysis_dir(REPO_ROOT)
assert ANALYSIS_DIR is not None, f'analysis/ not found anywhere under {REPO_ROOT}'

DATA_DIR = os.path.join(ANALYSIS_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

print(f'Repo root:    {REPO_ROOT}')
print(f'Analysis dir: {ANALYSIS_DIR}')
print(f'Data dir:     {DATA_DIR}')
print(f'Platform:     {platform.system()} {platform.machine()}')
print('Tip: edit C files in Colab and rerun build cell to apply baked-in parameter changes.')


In [ ]:
# Build the shared library from the cloned repo
os.chdir(ANALYSIS_DIR)
ret = os.system('make clean && make')
assert ret == 0, 'Build  check compiler output above'failed 

# Determine library path
if platform.system() == 'Darwin':
    LIB_PATH = os.path.join(ANALYSIS_DIR, 'libkalman.dylib')
else:
    LIB_PATH = os.path.join(ANALYSIS_DIR, 'libkalman.so')

assert os.path.exists(LIB_PATH), f'Library not found at {LIB_PATH}'
print(f'Library built: {LIB_PATH}')
print('If you change kalman_core.c or headers, rerun this cell before analysis.')


## 2. ctypes Bindings

In [ ]:
import ctypes
from ctypes import c_float, c_uint8, c_uint16, c_uint32, c_bool, POINTER, Structure, byref
import numpy as np

# --- Constants (must match kalman_core.h) ---
KC_STATE_DIM = 9

# State indices
KC_STATE_X  = 0  # Position X (world, m)
KC_STATE_Y  = 1  # Position Y (world, m)
KC_STATE_Z  = 2  # Position Z (world, m)
KC_STATE_PX = 3  # Velocity X (body, m/s)
KC_STATE_PY = 4  # Velocity Y (body, m/s)
KC_STATE_PZ = 5  # Velocity Z (body, m/s)
KC_STATE_D0 = 6  # Attitude error roll (rad)
KC_STATE_D1 = 7  # Attitude error pitch (rad)
KC_STATE_D2 = 8  # Attitude error yaw (rad)


# --- C struct mirrors ---

class ArmMatrixInstanceF32(Structure):
    """Mirrors arm_matrix_instance_f32"""
    _fields_ = [
        ('numRows', c_uint16),
        ('numCols', c_uint16),
        ('pData', POINTER(c_float)),
    ]


class Axis3f(Structure):
    """Mirrors Axis3f (3-axis float vector)"""
    _fields_ = [
        ('x', c_float),
        ('y', c_float),
        ('z', c_float),
    ]


class DistanceMeasurement(Structure):
    """Mirrors distanceMeasurement_t"""
    _fields_ = [
        ('x', c_float),
        ('y', c_float),
        ('z', c_float),
        ('distance', c_float),
        ('stdDev', c_float),
        ('anchorId', c_uint8),
    ]


class KalmanCoreParams(Structure):
    """Mirrors kalmanCoreParams_t"""
    _fields_ = [
        ('stdDevInitialPosition_xy', c_float),
        ('stdDevInitialPosition_z', c_float),
        ('stdDevInitialVelocity', c_float),
        ('stdDevInitialAttitude_rollpitch', c_float),
        ('stdDevInitialAttitude_yaw', c_float),
        ('procNoiseAcc_xy', c_float),
        ('procNoiseAcc_z', c_float),
        ('procNoiseVel', c_float),
        ('procNoisePos', c_float),
        ('procNoiseAtt', c_float),
        ('measNoiseGyro_rollpitch', c_float),
        ('measNoiseGyro_yaw', c_float),
        ('initialX', c_float),
        ('initialY', c_float),
        ('initialZ', c_float),
        ('initialYaw', c_float),
    ]


# Covariance matrix type: float[9][9]
CovMatrix = (c_float * KC_STATE_DIM) * KC_STATE_DIM
# Rotation matrix type: float[3][3]
RotMatrix = (c_float * 3) * 3


class KalmanCoreData(Structure):
    """Mirrors kalmanCoreData_t"""
    _fields_ = [
        ('S', c_float * KC_STATE_DIM),         # State vector
        ('q', c_float * 4),                     # Quaternion [w, x, y, z]
        ('R', RotMatrix),                       # Rotation matrix (body to world)
        ('P', CovMatrix),                       # Covariance matrix (9x9)
        ('Pm', ArmMatrixInstanceF32),           # ARM matrix instance for P
        ('initialQuaternion', c_float * 4),     # Initial quaternion
        ('isUpdated', c_bool),                  # Update flag
        ('lastPredictionMs', c_uint32),         # Last prediction timestamp
        ('lastProcessNoiseUpdateMs', c_uint32), # Last process noise timestamp
    ]


print(f'KalmanCoreData size: {ctypes.sizeof(KalmanCoreData)} bytes')
print(f'KalmanCoreParams size: {ctypes.sizeof(KalmanCoreParams)} bytes')

In [ ]:
# --- Load shared library and declare function signatures ---

lib = ctypes.CDLL(LIB_PATH)

# void kalmanCoreDefaultParams(kalmanCoreParams_t* params)
lib.kalmanCoreDefaultParams.argtypes = [POINTER(KalmanCoreParams)]
lib.kalmanCoreDefaultParams.restype = None

# void kalmanCoreInit(kalmanCoreData_t* kf, const kalmanCoreParams_t* params, uint32_t nowMs)
lib.kalmanCoreInit.argtypes = [POINTER(KalmanCoreData), POINTER(KalmanCoreParams), c_uint32]
lib.kalmanCoreInit.restype = None

# void kalmanCorePredict(kalmanCoreData_t* kf, const kalmanCoreParams_t* params,
#                        Axis3f* acc, Axis3f* gyro, uint32_t nowMs)
lib.kalmanCorePredict.argtypes = [
    POINTER(KalmanCoreData), POINTER(KalmanCoreParams),
    POINTER(Axis3f), POINTER(Axis3f), c_uint32
]
lib.kalmanCorePredict.restype = None

# void kalmanCoreAddProcessNoise(kalmanCoreData_t* kf, const kalmanCoreParams_t* params,
#                                uint32_t nowMs)
lib.kalmanCoreAddProcessNoise.argtypes = [
    POINTER(KalmanCoreData), POINTER(KalmanCoreParams), c_uint32
]
lib.kalmanCoreAddProcessNoise.restype = None

# void kalmanCoreUpdateWithDistance(kalmanCoreData_t* kf, distanceMeasurement_t* d)
lib.kalmanCoreUpdateWithDistance.argtypes = [
    POINTER(KalmanCoreData), POINTER(DistanceMeasurement)
]
lib.kalmanCoreUpdateWithDistance.restype = None

# bool kalmanCoreFinalize(kalmanCoreData_t* kf)
lib.kalmanCoreFinalize.argtypes = [POINTER(KalmanCoreData)]
lib.kalmanCoreFinalize.restype = c_bool

# void kalmanCoreGetPosition(const kalmanCoreData_t* kf, float* x, float* y, float* z)
lib.kalmanCoreGetPosition.argtypes = [
    POINTER(KalmanCoreData), POINTER(c_float), POINTER(c_float), POINTER(c_float)
]
lib.kalmanCoreGetPosition.restype = None

# void kalmanCoreGetVelocity(const kalmanCoreData_t* kf, float* vx, float* vy, float* vz)
lib.kalmanCoreGetVelocity.argtypes = [
    POINTER(KalmanCoreData), POINTER(c_float), POINTER(c_float), POINTER(c_float)
]
lib.kalmanCoreGetVelocity.restype = None

# void kalmanCoreGetAttitude(const kalmanCoreData_t* kf, float* roll, float* pitch, float* yaw)
lib.kalmanCoreGetAttitude.argtypes = [
    POINTER(KalmanCoreData), POINTER(c_float), POINTER(c_float), POINTER(c_float)
]
lib.kalmanCoreGetAttitude.restype = None

# void kalmanCoreGetQuaternion(const kalmanCoreData_t* kf,
#                              float* qw, float* qx, float* qy, float* qz)
lib.kalmanCoreGetQuaternion.argtypes = [
    POINTER(KalmanCoreData),
    POINTER(c_float), POINTER(c_float), POINTER(c_float), POINTER(c_float)
]
lib.kalmanCoreGetQuaternion.restype = None

print('Library loaded and function signatures declared.')

## 3. Helper Functions

In [ ]:
def init_filter(params=None):
    """
    Initialize the Kalman filter with default or custom parameters.
    Returns (kf_data, kf_params) ctypes structs.
    """
    kf_params = KalmanCoreParams()
    lib.kalmanCoreDefaultParams(byref(kf_params))

    # Override with custom params if provided
    if params is not None:
        for field_name, _ in KalmanCoreParams._fields_:
            if field_name in params:
                setattr(kf_params, field_name, params[field_name])

    kf_data = KalmanCoreData()
    lib.kalmanCoreInit(byref(kf_data), byref(kf_params), c_uint32(0))

    return kf_data, kf_params


def get_state(kf_data):
    """Extract full state from filter as a dict."""
    x, y, z = c_float(), c_float(), c_float()
    vx, vy, vz = c_float(), c_float(), c_float()
    roll, pitch, yaw = c_float(), c_float(), c_float()
    qw, qx, qy, qz = c_float(), c_float(), c_float(), c_float()

    lib.kalmanCoreGetPosition(byref(kf_data), byref(x), byref(y), byref(z))
    lib.kalmanCoreGetVelocity(byref(kf_data), byref(vx), byref(vy), byref(vz))
    lib.kalmanCoreGetAttitude(byref(kf_data), byref(roll), byref(pitch), byref(yaw))
    lib.kalmanCoreGetQuaternion(byref(kf_data), byref(qw), byref(qx), byref(qy), byref(qz))

    # Extract covariance diagonal
    P_diag = [kf_data.P[i][i] for i in range(KC_STATE_DIM)]

    return {
        'x': x.value, 'y': y.value, 'z': z.value,
        'vx': vx.value, 'vy': vy.value, 'vz': vz.value,
        'roll': roll.value, 'pitch': pitch.value, 'yaw': yaw.value,
        'qw': qw.value, 'qx': qx.value, 'qy': qy.value, 'qz': qz.value,
        'P_diag': P_diag,
    }


def process_imu(kf_data, kf_params, timestamp_ms, ax, ay, az, gx, gy, gz):
    """Process one IMU measurement — predict + process noise + finalize."""
    acc = Axis3f(ax, ay, az)
    gyro = Axis3f(gx, gy, gz)
    ts = c_uint32(int(timestamp_ms))

    lib.kalmanCorePredict(byref(kf_data), byref(kf_params), byref(acc), byref(gyro), ts)
    lib.kalmanCoreAddProcessNoise(byref(kf_data), byref(kf_params), ts)
    lib.kalmanCoreFinalize(byref(kf_data))


def process_uwb(kf_data, anchor_x, anchor_y, anchor_z, distance, stddev, anchor_id):
    """Process one UWB range measurement — update + finalize."""
    d = DistanceMeasurement(
        x=anchor_x, y=anchor_y, z=anchor_z,
        distance=distance, stdDev=stddev, anchorId=int(anchor_id)
    )
    lib.kalmanCoreUpdateWithDistance(byref(kf_data), byref(d))
    lib.kalmanCoreFinalize(byref(kf_data))


print('Helper functions defined.')

## 4. Load CSV Data

Current functional event format uses mixed event rows.

Required columns:
- `timestamp_ms`: timestamp in milliseconds.
- `type`: event type (`IMU`, `RANGING`, or `POSITION`).

IMU columns:
- `accel_x`, `accel_y`, `accel_z` (m/s^2)
- `gyro_x`, `gyro_y`, `gyro_z` (rad/s)

RANGING columns:
- `dist_m` (preferred) or legacy `distance`
- `anchor_addr` (preferred) or legacy `anchor_id`
- `anchor_x`, `anchor_y`, `anchor_z`
- optional `stddev` (defaults to `DEFAULT_UWB_STDDEV` if missing)

POSITION columns (reference only):
- `pos_x`, `pos_y`, `pos_z`
- `vel_x`, `vel_y`, `vel_z`
- `confidence`

Offline fusion behavior in this notebook:
- Every `IMU` row runs EKF prediction.
- Every `RANGING` row runs EKF range update.
- `POSITION` rows are not fused (used for reference/visualization).

Place your CSV file in `analysis/data/` and set the filename below.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Set your CSV filename here ---
CSV_FILENAME = 'timeseriestrial1.csv'  # <-- change filename here
CSV_PATH = os.path.join(DATA_DIR, CSV_FILENAME)

if not os.path.exists(CSV_PATH):
    print(f'WARNING: CSV file not found at {CSV_PATH}')
    print('Place your data file in analysis/data/ and update CSV_FILENAME above.')
    print('Skipping data  you can still use the helper functions manually.')load 
    df = None
else:
    df = pd.read_csv(CSV_PATH)
    if 'timestamp_ms' in df.columns:
        df = df.sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    print(f'Loaded {len(df)} rows from {CSV_FILENAME}')
    print(f'Columns: {list(df.columns)}')

    if len(df) > 0 and 'timestamp_ms' in df.columns:
        print(f'Time range: {df["timestamp_ms"].min()} - {df["timestamp_ms"].max()} ms')

    event_counts = df['type'].astype(str).str.upper().value_counts() if 'type' in df.columns else pd.Series(dtype=int)
    print(f'IMU samples: {int(event_counts.get("IMU", 0))}')
    print(f'RANGING samples: {int(event_counts.get("RANGING", 0))}')
    print(f'POSITION samples: {int(event_counts.get("POSITION", 0))}')
    print(f'Legacy UWB samples: {int(event_counts.get("UWB", 0))}')
    display(df.head(10))

    # Quick visualization of fused POSITION events directly from the CSV
    pos_df = df[df['type'].astype(str).str.upper() == 'POSITION'].copy() if 'type' in df.columns else pd.DataFrame()
    if len(pos_df) > 0:
        pos_df = pos_df.dropna(subset=['pos_x', 'pos_y'])

    if len(pos_df) > 0:
        fig, ax = plt.subplots(1, 1, figsize=(7, 7))
        ax.plot(pos_df['pos_x'], pos_df['pos_y'], linewidth=1.0, alpha=0.85, label='POSITION')
        ax.plot(pos_df['pos_x'].iloc[0], pos_df['pos_y'].iloc[0], 'go', markersize=8, label='Start')
        ax.plot(pos_df['pos_x'].iloc[-1], pos_df['pos_y'].iloc[-1], 'rs', markersize=8, label='End')
        ax.set_xlabel('pos_x (m)')
        ax.set_ylabel('pos_y (m)')
        ax.set_title('CSV POSITION 2D Trajectory (XY)')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('No POSITION rows with valid pos_x/pos_y found for 2D plotting.')


## 5. Run the Filter

In [ ]:
# Default UWB/RANGING measurement std dev (matches firmware RANGING_DEFAULT_STDDEV_M)
DEFAULT_UWB_STDDEV = 0.2


def _get_numeric(row, candidates):
    """Return first finite numeric value found in candidate columns, else None."""
    for col in candidates:
        if col in row.index and pd.notna(row[col]):
            try:
                return float(row[col])
            except (TypeError, ValueError):
                continue
    return None


def _get_int(row, candidates):
    """Return first valid int found in candidate columns, else None."""
    for col in candidates:
        if col in row.index and pd.notna(row[col]):
            try:
                return int(float(row[col]))
            except (TypeError, ValueError):
                continue
    return None


def run_filter(df, custom_params=None, log_interval=1):
    """
    Run the EKF over the full dataset using the current event format.

    Processing behavior:
      - IMU      -> predict step
      - RANGING  -> range update step
      - POSITION -> reference only (ignored by filter loop)
    """
    kf_data, kf_params = init_filter(custom_params)

    if df is None or len(df) == 0:
        print('Input DataFrame is empty.')
        return pd.DataFrame()

    if 'timestamp_ms' not in df.columns or 'type' not in df.columns:
        raise ValueError("CSV must contain at least 'timestamp_ms' and 'type' columns.")

    ordered_df = df.sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    # Pre-allocate log lists
    log = {
        'timestamp_ms': [], 'type': [],
        'x': [], 'y': [], 'z': [],
        'vx': [], 'vy': [], 'vz': [],
        'roll': [], 'pitch': [], 'yaw': [],
        'qw': [], 'qx': [], 'qy': [], 'qz': [],
    }
    state_names = ['X', 'Y', 'Z', 'PX', 'PY', 'PZ', 'D0', 'D1', 'D2']
    for name in state_names:
        log[f'P_{name}'] = []

    n_rows = len(ordered_df)
    fused_count = 0
    skipped_counts = {'POSITION': 0, 'UNKNOWN': 0, 'INVALID_RANGING': 0, 'INVALID_IMU': 0}

    for idx, row in ordered_df.iterrows():
        ts = int(row['timestamp_ms'])
        mtype = str(row['type']).strip().upper()
        did_fuse = False

        if mtype == 'IMU':
            ax = _get_numeric(row, ['accel_x'])
            ay = _get_numeric(row, ['accel_y'])
            az = _get_numeric(row, ['accel_z'])
            gx = _get_numeric(row, ['gyro_x'])
            gy = _get_numeric(row, ['gyro_y'])
            gz = _get_numeric(row, ['gyro_z'])

            if None in (ax, ay, az, gx, gy, gz):
                skipped_counts['INVALID_IMU'] += 1
                continue

            process_imu(kf_data, kf_params, ts, ax, ay, az, gx, gy, gz)
            did_fuse = True

        elif mtype in ('RANGING', 'UWB'):
            dist = _get_numeric(row, ['dist_m', 'distance'])
            anchor_x = _get_numeric(row, ['anchor_x'])
            anchor_y = _get_numeric(row, ['anchor_y'])
            anchor_z = _get_numeric(row, ['anchor_z'])
            anchor_id = _get_int(row, ['anchor_addr', 'anchor_id'])
            stddev = _get_numeric(row, ['stddev'])
            if stddev is None:
                stddev = DEFAULT_UWB_STDDEV

            if None in (dist, anchor_x, anchor_y, anchor_z, anchor_id):
                skipped_counts['INVALID_RANGING'] += 1
                continue

            process_uwb(kf_data, anchor_x, anchor_y, anchor_z, dist, stddev, anchor_id)
            did_fuse = True

        elif mtype == 'POSITION':
            skipped_counts['POSITION'] += 1
            continue

        else:
            skipped_counts['UNKNOWN'] += 1
            continue

        if did_fuse:
            fused_count += 1

            # Log state snapshot at configured interval of fused measurements
            if fused_count % log_interval == 0:
                state = get_state(kf_data)
                log['timestamp_ms'].append(ts)
                log['type'].append(mtype)
                for key in ['x', 'y', 'z', 'vx', 'vy', 'vz',
                            'roll', 'pitch', 'yaw', 'qw', 'qx', 'qy', 'qz']:
                    log[key].append(state[key])
                for i, name in enumerate(state_names):
                    log[f'P_{name}'].append(state['P_diag'][i])

        if (idx + 1) % 5000 == 0:
            print(f'  Processed {idx + 1}/{n_rows} rows...')

    results = pd.DataFrame(log)
    print(f'Done. {len(results)} state snapshots logged from {fused_count} fused measurements.')
    print(f"Skipped POSITION rows: {skipped_counts['POSITION']}")
    print(f"Skipped invalid IMU rows: {skipped_counts['INVALID_IMU']}")
    print(f"Skipped invalid RANGING rows: {skipped_counts['INVALID_RANGING']}")
    print(f"Skipped unknown rows: {skipped_counts['UNKNOWN']}")
    return results


In [ ]:
# Run the filter on loaded data
if df is not None:
    results = run_filter(df)
    display(results.head())
else:
    print('No data loaded. Place CSV in analysis/data/ and re-run cell 4.')
    results = None

## 6. Visualization

In [ ]:
import matplotlib.pyplot as plt

def plot_results(results):
    """Generate standard analysis plots from filter results."""
    if results is None or len(results) == 0:
        print('No results to plot.')
        return

    t = (results['timestamp_ms'] - results['timestamp_ms'].iloc[0]) / 1000.0  # seconds

    fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

    # --- Position ---
    ax = axes[0]
    ax.plot(t, results['x'], label='X', linewidth=0.8)
    ax.plot(t, results['y'], label='Y', linewidth=0.8)
    ax.plot(t, results['z'], label='Z', linewidth=0.8)
    ax.set_ylabel('Position (m)')
    ax.set_title('Position Estimate')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Velocity ---
    ax = axes[1]
    ax.plot(t, results['vx'], label='Vx', linewidth=0.8)
    ax.plot(t, results['vy'], label='Vy', linewidth=0.8)
    ax.plot(t, results['vz'], label='Vz', linewidth=0.8)
    ax.set_ylabel('Velocity (m/s)')
    ax.set_title('Velocity Estimate (World Frame)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Attitude ---
    ax = axes[2]
    RAD2DEG = 180.0 / np.pi
    ax.plot(t, results['roll'] * RAD2DEG, label='Roll', linewidth=0.8)
    ax.plot(t, results['pitch'] * RAD2DEG, label='Pitch', linewidth=0.8)
    ax.plot(t, results['yaw'] * RAD2DEG, label='Yaw', linewidth=0.8)
    ax.set_ylabel('Angle (deg)')
    ax.set_title('Attitude Estimate (Euler Angles)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Covariance (position) ---
    ax = axes[3]
    ax.semilogy(t, results['P_X'], label='P_X', linewidth=0.8)
    ax.semilogy(t, results['P_Y'], label='P_Y', linewidth=0.8)
    ax.semilogy(t, results['P_Z'], label='P_Z', linewidth=0.8)
    ax.set_ylabel('Variance (m²)')
    ax.set_xlabel('Time (s)')
    ax.set_title('Position Covariance (Diagonal)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # --- 2D Trajectory ---
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.plot(results['x'], results['y'], linewidth=0.8, alpha=0.8)
    ax.plot(results['x'].iloc[0], results['y'].iloc[0], 'go', markersize=10, label='Start')
    ax.plot(results['x'].iloc[-1], results['y'].iloc[-1], 'rs', markersize=10, label='End')
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_title('2D Trajectory (XY Plane)')
    ax.set_aspect('equal')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


print('Plotting functions defined.')

In [ ]:
if results is not None:
    plot_results(results)

## 7. Covariance & Innovation Analysis

In [ ]:
def plot_covariance_all(results):
    """Plot all 9 covariance diagonal elements."""
    if results is None or len(results) == 0:
        print('No results to plot.')
        return

    t = (results['timestamp_ms'] - results['timestamp_ms'].iloc[0]) / 1000.0
    state_names = ['X', 'Y', 'Z', 'PX', 'PY', 'PZ', 'D0', 'D1', 'D2']
    labels = ['Pos X', 'Pos Y', 'Pos Z', 'Vel X', 'Vel Y', 'Vel Z',
              'Att Roll', 'Att Pitch', 'Att Yaw']

    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

    # Position covariance
    for i in range(3):
        axes[0].semilogy(t, results[f'P_{state_names[i]}'], label=labels[i], linewidth=0.8)
    axes[0].set_ylabel('Variance')
    axes[0].set_title('Position Covariance')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Velocity covariance
    for i in range(3, 6):
        axes[1].semilogy(t, results[f'P_{state_names[i]}'], label=labels[i], linewidth=0.8)
    axes[1].set_ylabel('Variance')
    axes[1].set_title('Velocity Covariance')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Attitude covariance
    for i in range(6, 9):
        axes[2].semilogy(t, results[f'P_{state_names[i]}'], label=labels[i], linewidth=0.8)
    axes[2].set_ylabel('Variance')
    axes[2].set_xlabel('Time (s)')
    axes[2].set_title('Attitude Error Covariance')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


if results is not None:
    plot_covariance_all(results)

## 8. Parameter Tuning Experiments

Use this section to compare filter behavior with different parameter settings.

In [ ]:
def compare_params(df, param_sets, labels):
    """
    Run the filter with multiple parameter sets and overlay results.

    Args:
        df: Input DataFrame
        param_sets: List of dicts (None = defaults)
        labels: List of label strings
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for params, label in zip(param_sets, labels):
        print(f'Running: {label}...')
        res = run_filter(df, custom_params=params, log_interval=1)
        t = (res['timestamp_ms'] - res['timestamp_ms'].iloc[0]) / 1000.0

        axes[0, 0].plot(t, res['x'], label=label, linewidth=0.8, alpha=0.8)
        axes[0, 1].plot(t, res['y'], label=label, linewidth=0.8, alpha=0.8)
        axes[1, 0].plot(res['x'], res['y'], label=label, linewidth=0.8, alpha=0.8)
        axes[1, 1].semilogy(t, res['P_X'], label=label, linewidth=0.8, alpha=0.8)

    axes[0, 0].set_title('X Position'); axes[0, 0].set_ylabel('m'); axes[0, 0].legend()
    axes[0, 1].set_title('Y Position'); axes[0, 1].set_ylabel('m'); axes[0, 1].legend()
    axes[1, 0].set_title('XY Trajectory'); axes[1, 0].set_aspect('equal'); axes[1, 0].legend()
    axes[1, 1].set_title('P_X Covariance'); axes[1, 1].set_ylabel('Variance'); axes[1, 1].legend()

    for ax in axes.flat:
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Example: compare default vs higher process noise
# Uncomment and modify to run:
# if df is not None:
#     compare_params(df,
#         param_sets=[
#             None,  # defaults
#             {'procNoiseAcc_xy': 1.0, 'procNoiseVel': 0.1},  # more IMU uncertainty
#             {'procNoiseAcc_xy': 0.1, 'procNoiseVel': 0.001},  # less IMU uncertainty
#         ],
#         labels=['Default', 'High Proc Noise', 'Low Proc Noise']
#     )

## 9. Export Results

In [ ]:
if results is not None:
    output_path = os.path.join(DATA_DIR, 'filter_output.csv')
    results.to_csv(output_path, index=False)
    print(f'Results exported to {output_path}')